# 13 — 从三元组到 R-GCN：知识图谱入门

上一课的 LightGCN 只知道节点之间“有边”，却不知道边为什么存在。本课加入**关系类型**。目标不是背模型名，而是逐步回答：数据长什么样、分数如何计算、负样本如何产生、R-GCN 比 embedding 基线多做了什么，以及 MRR/Hits@K 从哪里来。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.knowledge_graph import (
    load_fb15k237, make_toy_knowledge_graph, sample_corrupted_triples,
)
from crosscity.models.knowledge_graph import DistMult, RGCNDistMult, TransE
from crosscity.training.knowledge_graph import (
    filtered_ranking_metrics, train_knowledge_graph_model,
)
from crosscity.utils import seed_everything
if torch.cuda.is_available(): device = torch.device('cuda')
elif torch.backends.mps.is_available(): device = torch.device('mps')
else: device = torch.device('cpu')
device

## 1. 什么是知识图谱？

知识图谱中的一条事实写成三元组：

$$ (h,r,t)=(\text{头实体},\text{关系},\text{尾实体}) $$

例如 `(Alice, lives_in, Paris)`。Alice 和 Paris 是**实体**，`lives_in` 是有方向、有语义的**关系**。普通图只有 $A_{ij}=1$；知识图谱还要回答是哪种关系 $r$。因此 `(Paris, lives_in, Alice)` 与原事实完全不同。

推荐系统可写成 `(用户, 购买, 商品)`、`(商品, 属于, 类目)`、`(商品, 品牌, 公司)`。同一个节点能通过不同关系参与不同语义。

In [ ]:
data = make_toy_knowledge_graph()
def readable(triples):
    return pd.DataFrame([{
        'head': data.entity_names[int(h)],
        'relation': data.relation_names[int(r)],
        'tail': data.entity_names[int(t)],
    } for h, r, t in triples.t()])
print('entities:', data.num_entities, data.entity_names)
print('relations:', data.num_relations, data.relation_names)
readable(data.train)

### 张量如何保存三元组

`train` 的形状是 `[3, num_triples]`。第 0/1/2 行分别保存 head、relation、tail 的整数 ID。R-GCN 的传播图则使用 `edge_index[2, E]` 保存首尾实体，用 `edge_type[E]` 额外保存每条边的关系。

传播图只放训练事实。为了让消息也能从尾实体传回头实体，我们增加反向边，并把反向关系编号设为 $r+|\mathcal R|$。这只是传播通道，不会伪造新的监督标签。

In [ ]:
print('train triples shape:', tuple(data.train.shape))
print('message edge_index shape:', tuple(data.edge_index.shape))
print('message edge_type:', data.edge_type.tolist())
print('validation fact, invisible to propagation:')
readable(data.validation)

## 2. 知识图谱补全是什么任务？

给定不完整事实 `(Alice, lives_in, ?)`，模型需要给每个候选实体打分：

$$s(\text{Alice},\text{lives\_in},e),\quad e\in\mathcal E$$

然后把 Paris 排到尽量靠前的位置。这仍然是链接预测，但比 GAE 多了关系 $r$。模型也可以补 head：`(?, located_in, France)`，或者预测关系；本课评价 head 和 tail 补全。

## 3. TransE：把关系理解成向量平移

每个实体和关系都有 $d$ 维向量。对真事实，希望：

$$\mathbf h+\mathbf r\approx\mathbf t$$

因此距离越小越好，分数取距离的负数：

$$s_{TransE}(h,r,t)=-\|\mathbf h+\mathbf r-\mathbf t\|_1$$

例如一维情况下 Alice=2、`lives_in`=3、Paris=5，距离 $|2+3-5|=0$，分数最高。若把尾实体换成 London=9，距离变成 4。TransE 直观且适合有方向的关系，但一个头实体通过同一关系连接多个尾实体时，要求多个尾向量都靠近同一点，表达能力会受限。

In [ ]:
toy_head = torch.tensor([2.0])
toy_relation = torch.tensor([3.0])
for name, tail in {'Paris': torch.tensor([5.0]), 'London': torch.tensor([9.0])}.items():
    score = -(toy_head + toy_relation - tail).abs().sum()
    print(name, 'TransE score =', score.item())

## 4. DistMult：让关系选择匹配维度

DistMult 使用逐维乘积：

$$s_{DistMult}(h,r,t)=\sum_{k=1}^{d}h_k r_k t_k$$

关系向量像一个维度开关：$r_k$ 大，表示该关系更重视第 $k$ 个潜在语义。若 head 与 tail 在重要维度方向一致，分数上升。它比 TransE 更像关系条件下的相似度。

但乘法满足 $h_k r_k t_k=t_k r_k h_k$，所以 DistMult 有 $s(h,r,t)=s(t,r,h)$。它适合 `friend_of` 等对称关系，却难以严格表达 `parent_of` 这类反对称关系。

In [ ]:
h = torch.tensor([1.0, 2.0]); r = torch.tensor([2.0, 0.1]); t = torch.tensor([3.0, 1.0])
print('dimension contributions:', h * r * t)
print('DistMult score:', (h * r * t).sum().item())
print('swap head/tail:', (t * r * h).sum().item())

## 5. 负三元组与训练损失

数据只给出真事实，没有直接给假事实。我们随机替换 head 或 tail：

```text
(Alice, lives_in, Paris)     正例
(Alice, lives_in, UK)        替换 tail
(London, lives_in, Paris)    替换 head
```

采样时必须排除所有 train/validation/test 中的已知真事实，否则会把真事实错误标成负例。逻辑损失为：

$$\mathcal L=softplus(-s^+)+softplus(s^-)$$

当正例分数 $s^+$ 增大时第一项下降；负例分数 $s^-$ 变小时第二项下降。这里仍有知识图谱不完整造成的假负例风险。

In [ ]:
negative = sample_corrupted_triples(
    data.train, data.num_entities, data.all_true_triples,
    generator=torch.Generator().manual_seed(42),
)
pd.concat({'positive': readable(data.train), 'corrupted negative': readable(negative)})

## 6. R-GCN：不同关系使用不同消息变换

普通 GCN 对所有邻居使用同一种变换；R-GCN 按关系分组：

$$\mathbf h_i^{(l+1)}=\sigma\left(W_0\mathbf h_i^{(l)}+\sum_{r\in\mathcal R}\sum_{j\in\mathcal N_r(i)}\frac{1}{c_{i,r}}W_r\mathbf h_j^{(l)}\right)$$

逐项解释：$W_0h_i$ 保留自身信息；$\mathcal N_r(i)$ 是通过关系 $r$ 连到 $i$ 的邻居；$W_r$ 是关系专属变换；$c_{i,r}$ 做邻居数量归一化。于是 `lives_in` 和 `friend_of` 的消息不会被当成同一种含义。

如果有 1000 种关系，每种都存一个 $d\times d$ 矩阵会很贵。基分解令 $W_r=\sum_b a_{rb}V_b$：少量共享基矩阵 $V_b$ 表达通用变换，每个关系只学习组合系数 $a_{rb}$。本课用 R-GCN 编码实体，再用 DistMult 解码三元组。

## 7. MRR 与 Hits@K 是怎么来的？

假设补全 `(Alice, lives_in, ?)` 时 Paris 在全部实体中排第 2，则 reciprocal rank 是 $1/2$。对所有 head/tail 查询平均：

$$MRR=\frac{1}{Q}\sum_{q=1}^{Q}\frac{1}{rank_q}$$

Hits@K 是正确实体进入前 K 的查询比例。Hits@1 最严格；Hits@10 更宽松。

**Filtered evaluation** 很重要：若 `(Alice, lives_in, Paris)` 和 `(Alice, lives_in, London)` 都是真事实，评价 Paris 时必须把 London 从竞争者中移除，否则模型会因为把另一个真事实排得更高而被错误惩罚。

In [ ]:
# 微型图很小，结果用于理解机制，不用于宣称哪种架构普遍更强。
models = {
    'TransE': lambda: TransE(data.num_entities, data.num_relations, 16),
    'DistMult': lambda: DistMult(data.num_entities, data.num_relations, 16),
    'RGCN+DistMult': lambda: RGCNDistMult(
        data.num_entities, data.num_relations, 16, num_bases=3
    ),
}
rows, histories = [], {}
for seed in (42, 43, 44):
    for name, build in models.items():
        seed_everything(seed)
        model = build()
        result = train_knowledge_graph_model(
            model, data, device=device, seed=seed, max_epochs=400,
            patience=100, evaluation_interval=5,
        )
        rows.append({
            'seed': seed, 'model': name, 'best_epoch': result.best_epoch,
            'validation_mrr': result.validation.mean_reciprocal_rank,
            'test_mrr': result.test.mean_reciprocal_rank,
            'hits@1': result.test.hits_at_1, 'hits@3': result.test.hits_at_3,
            'hits@10': result.test.hits_at_10,
        })
        histories[(seed, name)] = result.history
results = pd.DataFrame(rows)
display(results)
results.groupby('model')[['validation_mrr', 'test_mrr', 'hits@1', 'hits@3']].agg(['mean', 'std'])

In [ ]:
for (seed, name), history in histories.items():
    if seed == 42:
        frame = pd.DataFrame(history)
        plt.plot(frame.epoch, frame.validation_mrr, label=name)
plt.xlabel('epoch'); plt.ylabel('filtered validation MRR')
plt.title('Toy KG: embedding baseline vs relation-aware propagation')
plt.legend(); plt.show()

## 8. 公开数据 FB15k-237（可选）

FB15k-237 包含约 1.45 万实体、237 种关系和数十万三元组。它需要联网首次下载。默认只在 `RUN_PUBLIC=True` 时加载并打印形状，因为对全部验证三元组、全部实体做 filtered ranking 需要专门的分批训练和分块评价，不能直接把微型教学循环冒充成高效基准实现。下一课会专门解决知识图谱的大规模训练与异构推荐。

In [ ]:
RUN_PUBLIC = False
if RUN_PUBLIC:
    fb = load_fb15k237(ROOT / 'data/raw/fb15k237')
    print('entities:', fb.num_entities, 'relations:', fb.num_relations)
    print('train/validation/test:', fb.train.size(1), fb.validation.size(1), fb.test.size(1))
else:
    print('保持 False 可跳过下载；理解本课公式不依赖公开数据。')

## 本课结论

- 知识图谱的基本单位是有方向、有类型的三元组，而不是无类型边。
- TransE 与 DistMult 不传播消息，是必须保留的 embedding 基线。
- R-GCN 的价值来自关系专属的消息变换，不只是传播得更远。
- 损坏三元组只是采样负例，未必是现实中的绝对假事实。
- MRR 衡量正确实体的平均倒数排名，Hits@K 衡量进入前 K 的比例。
- filtered evaluation 会排除其他已知真事实，是知识图谱补全的标准关键步骤。
- 小图实验用于检查机制；公开基准结论必须使用完整数据、分批训练、多 seed 和严格评价。